# Spatial router: только выход B0 → выбрать B5-плитки

Основано на идее *Sparse Refinement for Efficient High-Resolution Semantic
Segmentation* (ECCV 2024): дешёвое плотное предсказание выбирает небольшое
число мест для дорогого уточнения. Это не очередной fusion B0/B5: тяжёлая B5
уже посчитана и не запускается. Обучается только компактный router на P0.

**Вход router для одной плитки** — пространственная карта B0 размером 16×16:
19 вероятностей классов, entropy, разница top-1/top-2, карта границ B0 и x/y;
плюс глобальный контекст кадра: средние 19 вероятностей и средняя entropy.
Router предсказывает ожидаемый выигрыш B5 над B0 в плитке. P1 не используется
до OOF-защиты на P0.



In [2]:
# ACDC из папки «Доступные мне» → временный диск Colab /content.
# Не использует место MyDrive и не хранит вручную прописанные ID файлов.

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth
from pathlib import Path
import shutil

credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials)

# ID папки из вашей ссылки:
FOLDER_ID = "13H2OWdHD2plemZhMoBfN3zR4b3nq-BHD"

# Показываем, под каким аккаунтом авторизован Colab.
account = drive_api.about().get(fields="user(emailAddress)").execute()
print("Colab авторизован как:", account["user"]["emailAddress"])

# Находим файлы непосредственно внутри общей папки.
response = drive_api.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed = false",
    fields="files(id,name,size,mimeType,capabilities(canDownload))",
    supportsAllDrives=True,
    includeItemsFromAllDrives=True,
).execute()

items = response.get("files", [])
print("\nФайлы, которые Colab видит в общей папке:")
for item in items:
    size_gb = int(item.get("size", 0)) / 1024**3
    print(f"  {item['name']} | {size_gb:.2f} GB | id={item['id']}")

by_name = {item["name"]: item for item in items}

assert "gt_trainval.zip" in by_name, (
    "Colab не видит gt_trainval.zip. Проверьте, что при авторизации "
    "выбрали тот же аккаунт, где папка видна в разделе «Доступные мне»."
)
assert "rgb_anon_trainvaltest.zip" in by_name, (
    "Colab не видит rgb_anon_trainvaltest.zip. Проверьте аккаунт авторизации."
)

ARCHIVE_DIR = Path("/content/acdc_archives")
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

def download_from_shared_drive(item):
    filename = item["name"]
    file_id = item["id"]
    destination = ARCHIVE_DIR / filename
    expected_size = int(item["size"])

    assert item.get("capabilities", {}).get("canDownload", False), (
        f"Нет права на скачивание: {filename}"
    )

    if destination.exists() and destination.stat().st_size == expected_size:
        print(f"\n✓ Уже скачан: {filename}")
        return destination

    if destination.exists():
        print(f"\nУдаляю неполную временную копию: {filename}")
        destination.unlink()

    free_gb = shutil.disk_usage("/content").free / 1024**3
    required_gb = expected_size / 1024**3 + 2
    assert free_gb > required_gb, (
        f"На /content свободно {free_gb:.1f} GB, нужно хотя бы {required_gb:.1f} GB."
    )

    print(f"\nСкачиваю {filename} в /content...")
    request = drive_api.files().get_media(
        fileId=file_id,
        supportsAllDrives=True,
    )

    last_bucket = -1
    with open(destination, "wb") as handle:
        downloader = MediaIoBaseDownload(
            handle,
            request,
            chunksize=64 * 1024 * 1024,
        )

        done = False
        while not done:
            status, done = downloader.next_chunk()
            percent = int(status.progress() * 100)

            if percent // 5 != last_bucket:
                last_bucket = percent // 5
                print(f"{percent}%")

    assert destination.stat().st_size == expected_size
    print(f"✓ Готово: {destination}")
    return destination

GT_ARCHIVE = download_from_shared_drive(by_name["gt_trainval.zip"])
RGB_ARCHIVE = download_from_shared_drive(by_name["rgb_anon_trainvaltest.zip"])

print("\n✓ ACDC-архивы готовы:")
print("GT: ", GT_ARCHIVE)
print("RGB:", RGB_ARCHIVE)

Colab авторизован как: pronastshalim@gmail.com

Файлы, которые Colab видит в общей папке:
  rgb_anon_trainvaltest.zip | 15.59 GB | id=1RauTG86LkGR1o64RWfDRG1mz_2loclVx
  gt_trainval.zip | 0.12 GB | id=1_SqW2pcVXq3cvUPlf0a9KuTQO5SlmirG

Скачиваю gt_trainval.zip в /content...
50%
100%
✓ Готово: /content/acdc_archives/gt_trainval.zip

Скачиваю rgb_anon_trainvaltest.zip в /content...
0%
5%
10%
15%
20%
25%
30%
35%
40%
45%
50%
55%
60%
65%
70%
75%
80%
85%
90%
95%
100%
✓ Готово: /content/acdc_archives/rgb_anon_trainvaltest.zip

✓ ACDC-архивы готовы:
GT:  /content/acdc_archives/gt_trainval.zip
RGB: /content/acdc_archives/rgb_anon_trainvaltest.zip


In [3]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next(
    filename for filename in uploaded
    if filename == "spatial_router_cityscapes_p0_share.zip"
)

ROUTER_DIR = Path("/content/spatial_router")
ROUTER_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(ROUTER_DIR)

ROUTER_PATH = ROUTER_DIR / "spatial_router_cityscapes_p0.pt"

assert ROUTER_PATH.exists(), f"Не найден роутер: {ROUTER_PATH}"
print("✓ Роутер готов:", ROUTER_PATH)

Saving spatial_router_cityscapes_p0_share.zip to spatial_router_cityscapes_p0_share.zip
✓ Роутер готов: /content/spatial_router/spatial_router_cityscapes_p0.pt


In [4]:
payload = torch.load(ROUTER_PATH, map_location=device)
router.load_state_dict(payload["state_dict"])

NameError: name 'torch' is not defined

In [5]:
# ACDC zero-shot: Cityscapes-trained SpatialRouter vs entropy.
# Никакого обучения в этой ячейке нет.

%pip -q install transformers pyarrow scipy pandas

import os
import sys
import json
import zipfile
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image

assert torch.cuda.is_available(), "В Colab включите GPU: Runtime → Change runtime type → T4 GPU."
device = "cuda"

# -------------------------------------------------------------------
# 1. Код проекта и пути к уже загруженным файлам
# -------------------------------------------------------------------
REPO = Path("/content/error-geometry-routing")

if not (REPO / "core").exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "--branch", "experiments/frozen-router-results-20261025",
            "https://github.com/Ottobiss/error-geometry-routing.git",
            str(REPO),
        ],
        check=True,
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

GT_ARCHIVE = Path("/content/acdc_archives/gt_trainval.zip")
RGB_ARCHIVE = Path("/content/acdc_archives/rgb_anon_trainvaltest.zip")
ROUTER_PATH = Path("/content/spatial_router/spatial_router_cityscapes_p0.pt")

assert GT_ARCHIVE.exists(), f"Не найден: {GT_ARCHIVE}"
assert RGB_ARCHIVE.exists(), f"Не найден: {RGB_ARCHIVE}"
assert ROUTER_PATH.exists(), f"Не найден router checkpoint: {ROUTER_PATH}"

# None = полный ACDC validation (406 кадров).
# Для быстрой проверки работы можно временно поставить 5 или 10.
MAX_PER_CONDITION = None

# -------------------------------------------------------------------
# 2. Распаковываем только validation-кадры ACDC, не train/test
# -------------------------------------------------------------------
RAW_ROOT = Path("/content/acdc_val_raw")
ACDC_ROOT = REPO / "data" / "acdc"
ACDC_ROOT.mkdir(parents=True, exist_ok=True)

def extract_val_only(archive_path, suffix):
    with zipfile.ZipFile(archive_path) as archive:
        members = [
            item for item in archive.infolist()
            if not item.is_dir()
            and "/val/" in item.filename
            and item.filename.endswith(suffix)
        ]

        assert members, f"В {archive_path.name} не найдены val-файлы {suffix}"
        print(f"{archive_path.name}: извлекаю {len(members)} validation-файлов")

        for item in members:
            target = RAW_ROOT / item.filename
            if not target.exists():
                archive.extract(item, RAW_ROOT)

def normalize_acdc_files(suffix, top_folder):
    source_files = list(RAW_ROOT.rglob(f"*{suffix}"))
    assert source_files, f"Не найдены распакованные файлы {suffix}"

    conditions = {"fog", "night", "rain", "snow"}

    for source in source_files:
        condition = next((part for part in source.parts if part in conditions), None)
        assert condition is not None, f"Не найдено condition в пути: {source}"

        sequence = source.parent.name
        destination = ACDC_ROOT / top_folder / condition / "val" / sequence / source.name
        destination.parent.mkdir(parents=True, exist_ok=True)

        if not destination.exists():
            shutil.copy2(source, destination)

extract_val_only(RGB_ARCHIVE, "_rgb_anon.png")
extract_val_only(GT_ARCHIVE, "_gt_labelTrainIds.png")

normalize_acdc_files("_rgb_anon.png", "rgb_anon")
normalize_acdc_files("_gt_labelTrainIds.png", "gt")

# -------------------------------------------------------------------
# 3. Импорты проекта, модель роутера и SegFormer B0/B5
# -------------------------------------------------------------------
from core.components import NC, IGNORE
from core.data import ACDC
from core.evaluation import _miou, _miou_batch, entropy_score
from core.models import load_model
from core.tiles import grid, tile_effects, tile_index, window

class SpatialRouter(nn.Module):
    def __init__(self, channels=24, global_dim=20):
        super().__init__()
        self.local = nn.Sequential(
            nn.Conv2d(channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.score = nn.Sequential(
            nn.Linear(32 + global_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x, g):
        return self.score(torch.cat([self.local(x), g], dim=1)).squeeze(1)

payload = torch.load(ROUTER_PATH, map_location="cpu")
router = SpatialRouter(
    channels=payload.get("channels", 24),
    global_dim=payload.get("global_dim", 20),
).to(device)

router.load_state_dict(payload["state_dict"])
router.eval()

print("✓ Router:", ROUTER_PATH)
print("✓ Router trained on:", payload.get("trained_on", "Cityscapes P0"))

ds = ACDC(full=False)
assert len(ds) >= 400, f"Ожидалось около 406 ACDC val кадров, найдено: {len(ds)}"

# При необходимости берём небольшую равномерную подвыборку.
selected_indices = []
used_per_condition = {condition: 0 for condition in ACDC.CONDITIONS}

for index, image_id in enumerate(ds.image_ids):
    condition = image_id.split("_", 1)[0]

    if MAX_PER_CONDITION is None or used_per_condition[condition] < MAX_PER_CONDITION:
        selected_indices.append(index)
        used_per_condition[condition] += 1

print("Кадров для ACDC-теста:", len(selected_indices))
print("По условиям:", used_per_condition)

model_b0 = load_model("b0", device=device)
model_b5 = load_model("b5", device=device)

# -------------------------------------------------------------------
# 4. Признаки B0 — ровно та же схема, что была при обучении роутера
# -------------------------------------------------------------------
TILE = int(payload.get("tile", 256))
MARGIN = int(payload.get("margin", 128))
MAP = 16
SIZE = ds.size                      # (1024, 576)
boxes = grid(SIZE, TILE)
tidx = tile_index(SIZE, TILE)
n_tiles = len(boxes)

assert n_tiles == 12, f"Для ACDC ожидается 12 тайлов, найдено {n_tiles}"

def boundary_map(prediction):
    edge = np.zeros(prediction.shape, dtype=np.float32)
    edge[:, :-1] = np.maximum(edge[:, :-1], prediction[:, :-1] != prediction[:, 1:])
    edge[:-1, :] = np.maximum(edge[:-1, :], prediction[:-1, :] != prediction[1:, :])
    return edge

@torch.inference_mode()
def router_inputs_and_b0(sample):
    probabilities = model_b0.probs(sample, SIZE, full=False)
    pred_b0 = probabilities.argmax(0).cpu().numpy().astype(np.uint8)

    entropy = entropy_score(probabilities)[None]
    top2 = probabilities.topk(2, dim=0).values
    margin = (top2[0] - top2[1])[None]

    edge = torch.from_numpy(boundary_map(pred_b0))[None].to(device)

    yy, xx = torch.meshgrid(
        torch.linspace(0, 1, SIZE[1], device=device),
        torch.linspace(0, 1, SIZE[0], device=device),
        indexing="ij",
    )

    # 19 class probabilities + entropy + margin + edge + x/y = 24 channels.
    maps = torch.cat([
        probabilities,
        entropy,
        margin,
        edge,
        xx[None],
        yy[None],
    ])

    global_context = torch.cat([
        probabilities.mean((1, 2)),
        entropy.mean((1, 2)),
    ])

    tile_x, tile_entropy = [], []

    for x0, y0, x1, y1 in boxes:
        tile_x.append(
            F.interpolate(
                maps[:, y0:y1, x0:x1][None],
                size=(MAP, MAP),
                mode="bilinear",
                align_corners=False,
            )[0]
        )
        tile_entropy.append(entropy[0, y0:y1, x0:x1].mean())

    x = torch.stack(tile_x)
    g = global_context[None].repeat(n_tiles, 1)

    router_scores = router(x, g).cpu().numpy()
    entropy_scores = torch.stack(tile_entropy).cpu().numpy()

    return pred_b0, router_scores, entropy_scores

def select_top_k(scores, k):
    return np.argsort(-scores, kind="stable")[:k]

@torch.inference_mode()
def b5_on_selected_tiles(sample, pred_b0, selected_tiles):
    """B5 запускается только на объединении тайлов, нужных entropy и router."""
    result = pred_b0.copy()
    source = sample["small"].to(device)
    side = TILE + 2 * MARGIN

    for tile_id in selected_tiles:
        x0, y0, x1, y1 = boxes[int(tile_id)]
        wx, wy = window((x0, y0, x1, y1), TILE, MARGIN, SIZE)

        crop = source[:, wy:wy + side, wx:wx + side][None]
        logits_b5 = model_b5(crop, (side, side))[0]
        prediction_b5 = logits_b5.argmax(0).cpu().numpy().astype(np.uint8)

        result[y0:y1, x0:x1] = prediction_b5[
            y0 - wy:y1 - wy,
            x0 - wx:x1 - wx,
        ]

    return result

# -------------------------------------------------------------------
# 5. ACDC evaluation: entropy vs router, budgets 1/2/3 B5-tiles
# -------------------------------------------------------------------
BUDGETS = (1, 2, 3)
per_group = {
    group: {
        k: {"entropy": [], "spatial B0 router": []}
        for k in BUDGETS
    }
    for group in (*ACDC.CONDITIONS, "all")
}

for number, dataset_index in enumerate(selected_indices, 1):
    sample = ds[dataset_index]
    image_id = sample["image_id"]
    condition = image_id.split("_", 1)[0]

    gt = sample["label"].numpy().astype(np.uint8)
    pred_b0, router_scores, entropy_scores = router_inputs_and_b0(sample)

    union = set()
    for k in BUDGETS:
        union.update(select_top_k(router_scores, k).tolist())
        union.update(select_top_k(entropy_scores, k).tolist())

    # B5 считается только на максимум шести тайлах, а не на всём изображении.
    pred_b5_union = b5_on_selected_tiles(
        sample,
        pred_b0,
        sorted(union),
    )

    effects, b0_confusion, _ = tile_effects(
        gt,
        pred_b0,
        pred_b5_union,
        tidx,
        n_tiles,
    )

    for k in BUDGETS:
        entropy_confusion = (
            b0_confusion
            + effects[select_top_k(entropy_scores, k)].sum(0)
        )
        router_confusion = (
            b0_confusion
            + effects[select_top_k(router_scores, k)].sum(0)
        )

        for group in (condition, "all"):
            per_group[group][k]["entropy"].append(entropy_confusion)
            per_group[group][k]["spatial B0 router"].append(router_confusion)

    if number % 5 == 0 or number == len(selected_indices):
        print(f"ACDC: {number}/{len(selected_indices)} кадров")

# -------------------------------------------------------------------
# 6. Итоговая таблица и paired bootstrap
# -------------------------------------------------------------------
def report_for_group(group, seed=0):
    rows = []
    rng = np.random.default_rng(seed)

    for k in BUDGETS:
        baseline = np.stack(per_group[group][k]["entropy"])
        routed = np.stack(per_group[group][k]["spatial B0 router"])

        n_frames = len(baseline)
        weights = rng.multinomial(
            n_frames,
            np.full(n_frames, 1 / n_frames),
            size=2000,
        )

        entropy_boot = _miou_batch(
            np.einsum("bn,nij->bij", weights, baseline)
        )
        router_boot = _miou_batch(
            np.einsum("bn,nij->bij", weights, routed)
        )
        difference = router_boot - entropy_boot
        ci_lo, ci_hi = np.percentile(difference, [2.5, 97.5])

        entropy_miou = _miou(baseline.sum(0))
        router_miou = _miou(routed.sum(0))

        rows.extend([
            {
                "domain": group,
                "n_frames": n_frames,
                "B5_patches_per_frame": k,
                "method": "entropy",
                "mIoU": entropy_miou,
                "delta_to_entropy": 0.0,
                "ci_lo": 0.0,
                "ci_hi": 0.0,
            },
            {
                "domain": group,
                "n_frames": n_frames,
                "B5_patches_per_frame": k,
                "method": "spatial B0 router",
                "mIoU": router_miou,
                "delta_to_entropy": router_miou - entropy_miou,
                "ci_lo": ci_lo,
                "ci_hi": ci_hi,
            },
        ])

    return rows

rows = []
for index, group in enumerate((*ACDC.CONDITIONS, "all")):
    if per_group[group][1]["entropy"]:
        rows.extend(report_for_group(group, seed=index))

report_acdc = pd.DataFrame(rows)
REPORT_PATH = Path("/content/acdc_router_report.csv")
report_acdc.to_csv(REPORT_PATH, index=False)

print("\n✓ ACDC zero-shot проверка завершена")
print("Таблица:", REPORT_PATH)
display(report_acdc)

rgb_anon_trainvaltest.zip: извлекаю 406 validation-файлов
gt_trainval.zip: извлекаю 406 validation-файлов
✓ Router: /content/spatial_router/spatial_router_cityscapes_p0.pt
✓ Router trained on: Cityscapes P0
Кадров для ACDC-теста: 406
По условиям: {'fog': 100, 'night': 106, 'rain': 100, 'snow': 100}


config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 15.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  339MB            

model.safetensors: downloading bytes:           |  0.00B            

ACDC: 5/406 кадров
ACDC: 10/406 кадров
ACDC: 15/406 кадров
ACDC: 20/406 кадров
ACDC: 25/406 кадров
ACDC: 30/406 кадров
ACDC: 35/406 кадров
ACDC: 40/406 кадров
ACDC: 45/406 кадров
ACDC: 50/406 кадров
ACDC: 55/406 кадров
ACDC: 60/406 кадров
ACDC: 65/406 кадров
ACDC: 70/406 кадров
ACDC: 75/406 кадров
ACDC: 80/406 кадров
ACDC: 85/406 кадров
ACDC: 90/406 кадров
ACDC: 95/406 кадров
ACDC: 100/406 кадров
ACDC: 105/406 кадров
ACDC: 110/406 кадров
ACDC: 115/406 кадров
ACDC: 120/406 кадров
ACDC: 125/406 кадров
ACDC: 130/406 кадров
ACDC: 135/406 кадров
ACDC: 140/406 кадров
ACDC: 145/406 кадров
ACDC: 150/406 кадров
ACDC: 155/406 кадров
ACDC: 160/406 кадров
ACDC: 165/406 кадров
ACDC: 170/406 кадров
ACDC: 175/406 кадров
ACDC: 180/406 кадров
ACDC: 185/406 кадров
ACDC: 190/406 кадров
ACDC: 195/406 кадров
ACDC: 200/406 кадров
ACDC: 205/406 кадров
ACDC: 210/406 кадров
ACDC: 215/406 кадров
ACDC: 220/406 кадров
ACDC: 225/406 кадров
ACDC: 230/406 кадров
ACDC: 235/406 кадров
ACDC: 240/406 кадров
ACDC: 245/40

,domain,n_frames,B5_patches_per_frame,method,mIoU,delta_to_entropy,ci_lo,ci_hi
0,fog,100,1,entropy,66.589319,0.000000,0.000000,0.000000
1,fog,100,1,spatial B0 router,67.313015,0.723696,0.219941,1.479058
2,fog,100,2,entropy,67.340580,0.000000,0.000000,0.000000
3,fog,100,2,spatial B0 router,70.711455,3.370874,-0.084105,4.666104
4,fog,100,3,entropy,71.435275,0.000000,0.000000,0.000000
5,fog,100,3,spatial B0 router,71.453126,0.017851,-0.243543,0.302783
6,night,106,1,entropy,25.007306,0.000000,0.000000,0.000000
7,night,106,1,spatial B0 router,24.629020,-0.378286,-0.766507,0.002844
8,night,106,2,entropy,27.736303,0.000000,0.000000,0.000000
9,night,106,2,spatial B0 router,27.158203,-0.578100,-1.629352,0.485036


In [6]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import shutil
import pandas as pd

SAVE_DIR = Path(
    "/content/drive/MyDrive/error-geometry-routing/"
    "acdc_zero_shot_20261028"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_SRC = Path("/content/acdc_router_report.csv")
ROUTER_SRC = Path("/content/spatial_router/spatial_router_cityscapes_p0.pt")
META_SRC = Path("/content/spatial_router/router_meta.json")

assert REPORT_SRC.exists(), f"Не найдена ACDC-таблица: {REPORT_SRC}"
assert ROUTER_SRC.exists(), f"Не найден checkpoint: {ROUTER_SRC}"

shutil.copy2(REPORT_SRC, SAVE_DIR / "acdc_router_report.csv")
shutil.copy2(ROUTER_SRC, SAVE_DIR / "spatial_router_cityscapes_p0.pt")

if META_SRC.exists():
    shutil.copy2(META_SRC, SAVE_DIR / "router_meta.json")

report = pd.read_csv(REPORT_SRC)
overall = report[
    (report["domain"] == "all")
    & (report["method"] == "spatial B0 router")
].copy()

summary = [
    "ACDC zero-shot: Cityscapes-trained SpatialRouter vs entropy",
    "",
    "Основной вывод:",
    "При бюджете 3 B5-патча роутер подтверждённо лучше entropy на ACDC all.",
    "",
    "Строки domain=all, spatial B0 router:",
    overall.to_string(index=False),
    "",
    "Правило применения:",
    "B5 budget >= 3  -> spatial B0 router",
    "B5 budget 1–2   -> entropy",
]
(SAVE_DIR / "README_results.txt").write_text(
    "\n".join(summary),
    encoding="utf-8",
)

print("✓ Результаты сохранены:")
for path in sorted(SAVE_DIR.iterdir()):
    print(" ", path.name, f"({path.stat().st_size / 1024:.1f} KB)")

print("\nПапка на Drive:", SAVE_DIR)

Mounted at /content/drive
✓ Результаты сохранены:
  README_results.txt (0.7 KB)
  acdc_router_report.csv (2.3 KB)
  router_meta.json (0.2 KB)
  spatial_router_cityscapes_p0.pt (73.9 KB)

Папка на Drive: /content/drive/MyDrive/error-geometry-routing/acdc_zero_shot_20261028


In [ ]:
from pathlib import Path
from google.colab import drive

MOUNT = Path('/content/gdrive')
if not (MOUNT / 'MyDrive').exists():
    drive.mount(str(MOUNT))
%pip -q install --upgrade-strategy only-if-needed transformers pyarrow



Mounted at /content/gdrive


In [ ]:
# Скачиваем переданную папку с данными сразу на ВАШ Google Drive.
# После успешного запуска повторно её выполнять не нужно.

!pip -q install --upgrade gdown

from pathlib import Path
from google.colab import drive
import gdown

MOUNT = Path("/content/drive")
if not (MOUNT / "MyDrive").exists():
    drive.mount("/content/drive")

DRIVE_DIR = MOUNT / "MyDrive" / "error-geometry-routing"
DATA_ON_DRIVE = DRIVE_DIR / "cityscapes"
DATA_ON_DRIVE.mkdir(parents=True, exist_ok=True)

# Важно: это ID из вашей ссылки — 1O8..., буква O после единицы, не цифра 0.
DATA_FOLDER_ID = "1O8_9tp_lj4ZHjKisqcUszGtdvFvXQp74"

already_have = list(DATA_ON_DRIVE.rglob("gtFine_trainvaltest.zip"))

if already_have:
    print("✓ Архив разметки уже есть:", already_have[0])
else:
    print("Скачиваю общую папку с данными на ваш Drive…")
    downloaded = gdown.download_folder(
        id=DATA_FOLDER_ID,
        output=str(DATA_ON_DRIVE),
        quiet=False,
        use_cookies=False,
    )
    print("Скачано файлов:", 0 if downloaded is None else len(downloaded))

archives = list(DATA_ON_DRIVE.rglob("gtFine_trainvaltest.zip"))
assert archives, (
    "В скачанной папке нет gtFine_trainvaltest.zip. "
    "Проверьте, что в исходной общей папке включён доступ «Anyone with the link»."
)

print("✓ Данные лежат на Drive:", DATA_ON_DRIVE)
print("✓ Разметка:", archives[0])

Скачиваю общую папку с данными на ваш Drive…


Retrieving folder contents


Processing file 1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk gtFine_trainvaltest.zip
Processing file 1yOszZjoa6dQdIoHS7CGo3EXm2Li-lRzC test.zip
Processing file 1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ val.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk
From (redirected): https://drive.google.com/uc?id=1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk&confirm=t&uuid=6be84181-f2ea-452e-99ea-92d5e2e8006c
To: /content/drive/MyDrive/error-geometry-routing/cityscapes/gtFine_trainvaltest.zip
100%|██████████| 253M/253M [00:02<00:00, 95.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1yOszZjoa6dQdIoHS7CGo3EXm2Li-lRzC
From (redirected): https://drive.google.com/uc?id=1yOszZjoa6dQdIoHS7CGo3EXm2Li-lRzC&confirm=t&uuid=f6281ee8-7fb5-4539-ac27-873a91194fea
To: /content/drive/MyDrive/error-geometry-routing/cityscapes/test.zip
100%|██████████| 3.46G/3.46G [00:46<00:00, 74.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ
From (redirected): https://drive.google.com/uc?id=1aErjLEt25oJEX48W-zxHwI

Скачано файлов: 3
✓ Данные лежат на Drive: /content/drive/MyDrive/error-geometry-routing/cityscapes
✓ Разметка: /content/drive/MyDrive/error-geometry-routing/cityscapes/gtFine_trainvaltest.zip



Download completed


In [ ]:
# ===== ЯЧЕЙКА 2 — подготовка B0, Cityscapes и сохранение прогресса =====
# Архивы уже скачаны на Google Drive.
# Ячейка сама находит их, распаковывает и исправляет вложенную структуру папок.

!pip -q install --upgrade-strategy only-if-needed cityscapesscripts

from pathlib import Path
import os
import sys
import shutil
import zipfile
import subprocess
import numpy as np

# ------------------------------------------------------------------
# 1. Пути
# ------------------------------------------------------------------
MOUNT = Path("/content/drive")

if not (MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

REPO = Path("/content/error-geometry-routing")
DRIVE_DIR = MOUNT / "MyDrive" / "error-geometry-routing"

# В этой папке уже лежат val.zip и gtFine_trainvaltest.zip.
DRIVE_ARCHIVES = DRIVE_DIR / "cityscapes"

# Постоянное хранилище результатов B0.
OUT = DRIVE_DIR / "spatial_b0_router_20261028"
SAVED_RESULTS = OUT / "baseline_results"

# Локальные файлы временного окружения Colab.
LOCAL_ARCHIVES = Path("/content/cityscapes-archives")
DATA_ROOT = REPO / "data" / "cityscapes"

LOCAL_ARCHIVES.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 2. Репозиторий
# ------------------------------------------------------------------
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Ottobiss/error-geometry-routing.git",
            str(REPO),
        ],
        check=True,
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Восстанавливаем результаты, если B0 уже рассчитывался ранее.
if SAVED_RESULTS.exists() and (SAVED_RESULTS / "cityscapes").exists():
    shutil.copytree(SAVED_RESULTS, REPO / "results", dirs_exist_ok=True)
    print("✓ Восстановлены сохранённые результаты:", SAVED_RESULTS)

# ------------------------------------------------------------------
# 3. Берём архивы с ВАШЕГО Google Drive
# ------------------------------------------------------------------
required_archives = {
    "val.zip": DRIVE_ARCHIVES / "val.zip",
    "gtFine_trainvaltest.zip": DRIVE_ARCHIVES / "gtFine_trainvaltest.zip",
}

for name, drive_file in required_archives.items():
    assert drive_file.exists(), f"Не найден файл на Drive: {drive_file}"

    local_file = LOCAL_ARCHIVES / name

    # Не копируем заново, если локальный архив уже полностью есть.
    if not local_file.exists() or local_file.stat().st_size != drive_file.stat().st_size:
        print(f"Копирую {name} с Drive…")
        shutil.copy2(drive_file, local_file)
    else:
        print(f"✓ Локальный {name} уже есть")

# ------------------------------------------------------------------
# 4. Распаковка
# ------------------------------------------------------------------
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def count_files(pattern):
    return len(list(DATA_ROOT.rglob(pattern)))

def unpack_if_missing(archive_name, pattern, expected_count):
    """Распаковывает только если нужных файлов ещё нет."""
    current = count_files(pattern)

    if current >= expected_count:
        print(f"✓ {archive_name} уже распакован: найдено {current} файлов")
        return

    archive = LOCAL_ARCHIVES / archive_name
    print(f"Распаковываю {archive_name}…")

    with zipfile.ZipFile(archive) as zf:
        zf.extractall(DATA_ROOT)

unpack_if_missing("val.zip", "*_leftImg8bit.png", 500)
unpack_if_missing("gtFine_trainvaltest.zip", "*_gtFine_labelIds.png", 500)

# ------------------------------------------------------------------
# 5. Исправление структуры после распаковки.
# Архив может содержать leftImg8bit/val напрямую либо внутри лишних папок.
# ------------------------------------------------------------------
def normalize_cityscapes_folder(folder_name, file_pattern):
    """
    Гарантирует структуру:
      data/cityscapes/<folder_name>/val/<city>/<file>
    независимо от внутренней структуры ZIP-архива.
    """
    expected = DATA_ROOT / folder_name
    expected_files = list(expected.rglob(file_pattern))

    if len(expected_files) == 500:
        return

    all_files = list(DATA_ROOT.rglob(file_pattern))
    assert len(all_files) >= 500, (
        f"После распаковки найдено только {len(all_files)} файлов {file_pattern}"
    )

    # Ищем папку с названием leftImg8bit / gtFine,
    # которая расположена глубже ожидаемой директории.
    candidates = []

    for file_path in all_files:
        for parent in file_path.parents:
            if parent.name == folder_name and parent != expected:
                candidates.append(parent)
                break

    candidates = list(dict.fromkeys(candidates))

    if candidates:
        source = max(
            candidates,
            key=lambda p: len(list(p.rglob(file_pattern))),
        )

        print(f"Исправляю структуру: {source} → {expected}")

        if not expected.exists():
            shutil.move(str(source), str(expected))
        else:
            # Если ожидаемая папка уже создана, объединяем содержимое.
            shutil.copytree(source, expected, dirs_exist_ok=True)

    # Проверка: если архив был устроен вообще без папки leftImg8bit/gtFine,
    # создаём ожидаемую структуру из найденных файлов.
    expected_files = list(expected.rglob(file_pattern))

    if len(expected_files) < 500:
        print(f"Собираю {folder_name}/val из найденных файлов…")

        for source_file in all_files:
            val_parent = next(
                (p for p in source_file.parents if p.name == "val"),
                None,
            )
            assert val_parent is not None, f"Не найден сегмент val в пути: {source_file}"

            city = source_file.parent.name
            destination = expected / "val" / city / source_file.name
            destination.parent.mkdir(parents=True, exist_ok=True)

            if not destination.exists():
                shutil.copy2(source_file, destination)

normalize_cityscapes_folder("leftImg8bit", "*_leftImg8bit.png")
normalize_cityscapes_folder("gtFine", "*_gtFine_labelIds.png")

images = list((DATA_ROOT / "leftImg8bit" / "val").rglob("*_leftImg8bit.png"))
labels = list((DATA_ROOT / "gtFine" / "val").rglob("*_gtFine_labelIds.png"))

assert len(images) == 500, (
    f"Найдено изображений val: {len(images)}; ожидалось 500. "
    f"Путь: {DATA_ROOT / 'leftImg8bit' / 'val'}"
)

assert len(labels) == 500, (
    f"Найдено разметок val: {len(labels)}; ожидалось 500. "
    f"Путь: {DATA_ROOT / 'gtFine' / 'val'}"
)

print(f"✓ Cityscapes готов: {len(images)} images, {len(labels)} labels")

# ------------------------------------------------------------------
# 6. Инференс лёгкой B0
# ------------------------------------------------------------------
from core.data import get_dataset
from core.models import load_model
from core.paths import DatasetResults

ds_b0 = get_dataset("cityscapes", full=False)
res_b0 = DatasetResults("cityscapes")

# Пропускаются все уже сохранённые кадры.
missing_indices = [
    index
    for index, image_id in enumerate(ds_b0.image_ids)
    if not res_b0.mask(image_id).exists()
]

print(f"B0 masks: готово {len(ds_b0) - len(missing_indices)}/{len(ds_b0)}")

if missing_indices:
    model_b0 = load_model("b0")
    print(f"Запускаю B0 на оставшихся кадрах: {len(missing_indices)}")

    drive_masks = SAVED_RESULTS / "cityscapes" / "masks"
    drive_masks.mkdir(parents=True, exist_ok=True)

    for count, index in enumerate(missing_indices, start=1):
        sample = ds_b0[index]
        image_id = sample["image_id"]

        probs = model_b0.probs(sample, ds_b0.size)
        pred_b0 = probs.argmax(dim=0).detach().cpu().numpy().astype(np.uint8)

        mask_path = res_b0.mask(image_id)
        mask_path.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            mask_path,
            pred_b0=pred_b0,
            pred_b2=pred_b0.copy(),  # техническая заглушка, B2 не запускается
        )

        # Сразу копируем каждый готовый кадр на Drive.
        # Даже при отключении GPU прогресс не потеряется.
        shutil.copy2(mask_path, drive_masks / mask_path.name)

        if count % 25 == 0 or count == len(missing_indices):
            print(f"B0: {count}/{len(missing_indices)}")

    print("✓ B0 masks сохранены на Google Drive")
else:
    print("✓ Все B0 masks уже рассчитаны")

# ------------------------------------------------------------------
# 7. Логиты B0 и кэши p0/p1
# ------------------------------------------------------------------
res = DatasetResults("cityscapes")

if not res.logits.exists():
    print("Считаю логиты B0…")
    subprocess.run(
        [sys.executable, "-m", "features.logits", "--dataset", "cityscapes"],
        cwd=REPO,
        check=True,
    )
    print("✓ Логиты B0 готовы")
else:
    print("✓ Логиты B0 уже есть")

from core.evaluation import routing_cache

for pop in ("p0", "p1"):
    if not res.cache(pop).exists():
        routing_cache("cityscapes", pop)
        print(f"✓ Построен cache_{pop}")
    else:
        print(f"✓ cache_{pop} уже есть")

# ------------------------------------------------------------------
# 8. Финальное сохранение
# ------------------------------------------------------------------
shutil.copytree(REPO / "results", SAVED_RESULTS, dirs_exist_ok=True)

print("\n✓ Вторая ячейка успешно завершена.")
print("Результаты сохранены здесь:", SAVED_RESULTS)

✓ Локальный val.zip уже есть
✓ Локальный gtFine_trainvaltest.zip уже есть
✓ val.zip уже распакован: найдено 500 файлов
✓ gtFine_trainvaltest.zip уже распакован: найдено 5000 файлов
Собираю leftImg8bit/val из найденных файлов…
✓ Cityscapes готов: 500 images, 500 labels


ModuleNotFoundError: No module named 'core'

In [ ]:
# ===== ПРОДОЛЖЕНИЕ: исправление пути к репозиторию и запуск B0 =====
# Датасет НЕ скачивается и НЕ распаковывается заново.

from pathlib import Path
import os
import sys
import shutil
import subprocess
import numpy as np

MOUNT = Path("/content/drive")
DRIVE_DIR = MOUNT / "MyDrive" / "error-geometry-routing"
OUT = DRIVE_DIR / "spatial_b0_router_20261028"
SAVED_RESULTS = OUT / "baseline_results"

# Здесь уже корректно подготовлены 500 изображений и 500 разметок.
OLD_DATA_ROOT = Path("/content/error-geometry-routing/data/cityscapes")

assert len(list((OLD_DATA_ROOT / "leftImg8bit" / "val").rglob("*_leftImg8bit.png"))) == 500
assert len(list((OLD_DATA_ROOT / "gtFine" / "val").rglob("*_gtFine_labelIds.png"))) == 500

# Эта папка гарантированно является ИСХОДНЫМ КОДОМ проекта.
REPO = Path("/content/error-geometry-routing-src")

if not (REPO / "core" / "data.py").exists():
    print("Клонирую исходный код проекта…")
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Ottobiss/error-geometry-routing.git",
            str(REPO),
        ],
        check=True,
    )

# Подключаем уже распакованный Cityscapes к правильной копии репозитория.
TARGET_DATA_ROOT = REPO / "data" / "cityscapes"
TARGET_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not TARGET_DATA_ROOT.exists():
    os.symlink(OLD_DATA_ROOT, TARGET_DATA_ROOT, target_is_directory=True)
    print("✓ Подключён уже распакованный Cityscapes")
else:
    print("✓ Cityscapes уже подключён к исходникам")

os.chdir(REPO)
sys.path = [p for p in sys.path if p != str(REPO)]
sys.path.insert(0, str(REPO))

# Восстанавливаем ранее сохранённые результаты, если они существуют.
if SAVED_RESULTS.exists() and (SAVED_RESULTS / "cityscapes").exists():
    shutil.copytree(SAVED_RESULTS, REPO / "results", dirs_exist_ok=True)
    print("✓ Восстановлены результаты B0 с Drive")

# ------------------------------------------------------------------
# B0 inference
# ------------------------------------------------------------------
from core.data import get_dataset
from core.models import load_model
from core.paths import DatasetResults

ds_b0 = get_dataset("cityscapes", full=False)
res_b0 = DatasetResults("cityscapes")

missing_indices = [
    index
    for index, image_id in enumerate(ds_b0.image_ids)
    if not res_b0.mask(image_id).exists()
]

print(f"B0 masks: готово {len(ds_b0) - len(missing_indices)}/{len(ds_b0)}")

if missing_indices:
    model_b0 = load_model("b0")
    drive_masks = SAVED_RESULTS / "cityscapes" / "masks"
    drive_masks.mkdir(parents=True, exist_ok=True)

    for count, index in enumerate(missing_indices, start=1):
        sample = ds_b0[index]
        image_id = sample["image_id"]

        probs = model_b0.probs(sample, ds_b0.size)
        pred_b0 = probs.argmax(dim=0).detach().cpu().numpy().astype(np.uint8)

        mask_path = res_b0.mask(image_id)
        mask_path.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            mask_path,
            pred_b0=pred_b0,
            pred_b2=pred_b0.copy(),
        )

        # Каждый готовый кадр сразу сохраняется на Drive.
        shutil.copy2(mask_path, drive_masks / mask_path.name)

        if count % 25 == 0 or count == len(missing_indices):
            print(f"B0: {count}/{len(missing_indices)}")

    print("✓ B0 masks сохранены на Drive")
else:
    print("✓ B0 уже полностью рассчитана")

# ------------------------------------------------------------------
# B0 logits
# ------------------------------------------------------------------
res = DatasetResults("cityscapes")

if not res.logits.exists():
    print("Считаю логиты B0…")
    subprocess.run(
        [sys.executable, "-m", "features.logits", "--dataset", "cityscapes"],
        cwd=REPO,
        check=True,
    )
    print("✓ Логиты B0 готовы")
else:
    print("✓ Логиты B0 уже есть")

# ------------------------------------------------------------------
# p0 / p1 cache
# ------------------------------------------------------------------
from core.evaluation import routing_cache

for pop in ("p0", "p1"):
    if not res.cache(pop).exists():
        routing_cache("cityscapes", pop)
        print(f"✓ Построен cache_{pop}")
    else:
        print(f"✓ cache_{pop} уже есть")

# Финальное сохранение.
shutil.copytree(REPO / "results", SAVED_RESULTS, dirs_exist_ok=True)

print("\n✓ B0 baseline полностью подготовлен.")
print("Результаты:", SAVED_RESULTS)

Клонирую исходный код проекта…
✓ Подключён уже распакованный Cityscapes


ModuleNotFoundError: No module named 'core'

In [ ]:
# ===== Исправление: берём именно ветку репозитория с исходным кодом =====
# Датасет Cityscapes уже подготовлен в старой папке и будет подключён без копирования.

!pip -q install --upgrade-strategy only-if-needed cityscapesscripts

from pathlib import Path
import os
import sys
import shutil
import subprocess
import numpy as np

# ---------------------------------------------------------------
# 1. Уже подготовленный Cityscapes
# ---------------------------------------------------------------
OLD_DATA_ROOT = Path("/content/error-geometry-routing/data/cityscapes")

n_images = len(list((OLD_DATA_ROOT / "leftImg8bit" / "val").rglob("*_leftImg8bit.png")))
n_labels = len(list((OLD_DATA_ROOT / "gtFine" / "val").rglob("*_gtFine_labelIds.png")))

assert n_images == 500 and n_labels == 500, (
    f"Ожидалось 500/500, найдено {n_images}/{n_labels}. "
    "Запустите прежнюю ячейку подготовки Cityscapes."
)

print(f"✓ Использую уже готовый Cityscapes: {n_images} images, {n_labels} labels")

# ---------------------------------------------------------------
# 2. Клонируем ВЕТКУ С КОДОМ, не пустую main-ветку
# ---------------------------------------------------------------
REMOTE = "https://github.com/Ottobiss/error-geometry-routing.git"

# Сначала пробуем экспериментальную ветку, затем исходный baseline.
BRANCHES = [
    "experiments/frozen-router-results-20261025",
    "baseline/baseline",
]

REPO = None

for branch in BRANCHES:
    safe_name = branch.replace("/", "-")
    candidate = Path(f"/content/error-geometry-routing-{safe_name}")

    if not (candidate / "core" / "data.py").exists():
        print(f"Клонирую ветку: {branch}")

        result = subprocess.run(
            [
                "git", "clone",
                "--depth", "1",
                "--branch", branch,
                REMOTE,
                str(candidate),
            ],
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print(f"Не удалось взять {branch}:")
            print(result.stderr[-800:])
            continue

    if (candidate / "core" / "data.py").exists():
        REPO = candidate
        print("✓ Найдена ветка с кодом:", branch)
        break

assert REPO is not None, (
    "Не удалось получить ветку с папкой core/. "
    "Скопируйте полный текст ошибки из ячейки."
)

# ---------------------------------------------------------------
# 3. Подключаем готовый датасет к правильной копии исходников
# ---------------------------------------------------------------
TARGET_DATA_ROOT = REPO / "data" / "cityscapes"
TARGET_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not TARGET_DATA_ROOT.exists():
    os.symlink(OLD_DATA_ROOT, TARGET_DATA_ROOT, target_is_directory=True)

assert len(list((TARGET_DATA_ROOT / "leftImg8bit" / "val").rglob("*_leftImg8bit.png"))) == 500
assert len(list((TARGET_DATA_ROOT / "gtFine" / "val").rglob("*_gtFine_labelIds.png"))) == 500

print("✓ Cityscapes подключён к исходникам:", TARGET_DATA_ROOT)

# ---------------------------------------------------------------
# 4. Настраиваем Python на правильный репозиторий
# ---------------------------------------------------------------
os.chdir(REPO)

for module_name in list(sys.modules):
    if module_name == "core" or module_name.startswith("core."):
        del sys.modules[module_name]

sys.path = [path for path in sys.path if "error-geometry-routing" not in path]
sys.path.insert(0, str(REPO))

# Это должно сработать: именно здесь раньше была ошибка.
from core.data import get_dataset
from core.models import load_model
from core.paths import DatasetResults

print("✓ Импорты core работают")

# ---------------------------------------------------------------
# 5. Пути сохранения и восстановление прогресса
# ---------------------------------------------------------------
MOUNT = Path("/content/drive")
DRIVE_DIR = MOUNT / "MyDrive" / "error-geometry-routing"
OUT = DRIVE_DIR / "spatial_b0_router_20261028"
SAVED_RESULTS = OUT / "baseline_results"

OUT.mkdir(parents=True, exist_ok=True)

if SAVED_RESULTS.exists() and (SAVED_RESULTS / "cityscapes").exists():
    shutil.copytree(SAVED_RESULTS, REPO / "results", dirs_exist_ok=True)
    print("✓ Восстановлены ранее сохранённые результаты B0")

# ---------------------------------------------------------------
# 6. B0 inference — только для ещё не готовых кадров
# ---------------------------------------------------------------
ds_b0 = get_dataset("cityscapes", full=False)
res_b0 = DatasetResults("cityscapes")

missing = [
    index
    for index, image_id in enumerate(ds_b0.image_ids)
    if not res_b0.mask(image_id).exists()
]

print(f"B0 masks: готово {len(ds_b0) - len(missing)}/{len(ds_b0)}")

if missing:
    model_b0 = load_model("b0")

    drive_masks = SAVED_RESULTS / "cityscapes" / "masks"
    drive_masks.mkdir(parents=True, exist_ok=True)

    for count, index in enumerate(missing, start=1):
        sample = ds_b0[index]
        image_id = sample["image_id"]

        probs = model_b0.probs(sample, ds_b0.size)
        pred_b0 = probs.argmax(dim=0).detach().cpu().numpy().astype(np.uint8)

        mask_file = res_b0.mask(image_id)
        mask_file.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            mask_file,
            pred_b0=pred_b0,
            pred_b2=pred_b0.copy(),
        )

        # Сохранение каждого готового кадра на Drive.
        shutil.copy2(mask_file, drive_masks / mask_file.name)

        if count % 25 == 0 or count == len(missing):
            print(f"B0: {count}/{len(missing)}")

    print("✓ B0 masks сохранены на Drive")
else:
    print("✓ B0 уже полностью рассчитана")

# ---------------------------------------------------------------
# 7. Логиты B0 и кэши для роутера
# ---------------------------------------------------------------
res = DatasetResults("cityscapes")

if not res.logits.exists():
    print("Считаю логиты B0…")
    subprocess.run(
        [sys.executable, "-m", "features.logits", "--dataset", "cityscapes"],
        cwd=REPO,
        check=True,
    )
    print("✓ Логиты B0 готовы")
else:
    print("✓ Логиты B0 уже есть")

from core.evaluation import routing_cache

for pop in ("p0", "p1"):
    if not res.cache(pop).exists():
        routing_cache("cityscapes", pop)
        print(f"✓ Построен cache_{pop}")
    else:
        print(f"✓ cache_{pop} уже существует")

shutil.copytree(REPO / "results", SAVED_RESULTS, dirs_exist_ok=True)

print("\n✓ Базовая подготовка завершена.")
print("Результаты сохранены на Drive:", SAVED_RESULTS)

✓ Использую уже готовый Cityscapes: 500 images, 500 labels
Клонирую ветку: experiments/frozen-router-results-20261025
✓ Найдена ветка с кодом: experiments/frozen-router-results-20261025
✓ Cityscapes подключён к исходникам: /content/error-geometry-routing-experiments-frozen-router-results-20261025/data/cityscapes
✓ Импорты core работают
B0 masks: готово 0/500


config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 15.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.9MB            

model.safetensors: downloading bytes:           |  0.00B            

B0: 25/500
B0: 50/500
B0: 75/500
B0: 100/500
B0: 125/500
B0: 150/500
B0: 175/500
B0: 200/500
B0: 225/500
B0: 250/500
B0: 275/500
B0: 300/500
B0: 325/500
B0: 350/500
B0: 375/500
B0: 400/500
B0: 425/500
B0: 450/500
B0: 475/500
B0: 500/500
✓ B0 masks сохранены на Drive
Считаю логиты B0…
✓ Логиты B0 готовы
✓ Построен cache_p0
✓ Построен cache_p1

✓ Базовая подготовка завершена.
Результаты сохранены на Drive: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/baseline_results


In [ ]:
# ===== Исправление: берём именно ветку репозитория с исходным кодом =====
# Датасет Cityscapes уже подготовлен в старой папке и будет подключён без копирования.

!pip -q install --upgrade-strategy only-if-needed cityscapesscripts

from pathlib import Path
import os
import sys
import shutil
import subprocess
import numpy as np

# ---------------------------------------------------------------
# 1. Уже подготовленный Cityscapes
# ---------------------------------------------------------------
OLD_DATA_ROOT = Path("/content/error-geometry-routing/data/cityscapes")

n_images = len(list((OLD_DATA_ROOT / "leftImg8bit" / "val").rglob("*_leftImg8bit.png")))
n_labels = len(list((OLD_DATA_ROOT / "gtFine" / "val").rglob("*_gtFine_labelIds.png")))

assert n_images == 500 and n_labels == 500, (
    f"Ожидалось 500/500, найдено {n_images}/{n_labels}. "
    "Запустите прежнюю ячейку подготовки Cityscapes."
)

print(f"✓ Использую уже готовый Cityscapes: {n_images} images, {n_labels} labels")

# ---------------------------------------------------------------
# 2. Клонируем ВЕТКУ С КОДОМ, не пустую main-ветку
# ---------------------------------------------------------------
REMOTE = "https://github.com/Ottobiss/error-geometry-routing.git"

# Сначала пробуем экспериментальную ветку, затем исходный baseline.
BRANCHES = [
    "experiments/frozen-router-results-20261025",
    "baseline/baseline",
]

REPO = None

for branch in BRANCHES:
    safe_name = branch.replace("/", "-")
    candidate = Path(f"/content/error-geometry-routing-{safe_name}")

    if not (candidate / "core" / "data.py").exists():
        print(f"Клонирую ветку: {branch}")

        result = subprocess.run(
            [
                "git", "clone",
                "--depth", "1",
                "--branch", branch,
                REMOTE,
                str(candidate),
            ],
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print(f"Не удалось взять {branch}:")
            print(result.stderr[-800:])
            continue

    if (candidate / "core" / "data.py").exists():
        REPO = candidate
        print("✓ Найдена ветка с кодом:", branch)
        break

assert REPO is not None, (
    "Не удалось получить ветку с папкой core/. "
    "Скопируйте полный текст ошибки из ячейки."
)

# ---------------------------------------------------------------
# 3. Подключаем готовый датасет к правильной копии исходников
# ---------------------------------------------------------------
TARGET_DATA_ROOT = REPO / "data" / "cityscapes"
TARGET_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not TARGET_DATA_ROOT.exists():
    os.symlink(OLD_DATA_ROOT, TARGET_DATA_ROOT, target_is_directory=True)

assert len(list((TARGET_DATA_ROOT / "leftImg8bit" / "val").rglob("*_leftImg8bit.png"))) == 500
assert len(list((TARGET_DATA_ROOT / "gtFine" / "val").rglob("*_gtFine_labelIds.png"))) == 500

print("✓ Cityscapes подключён к исходникам:", TARGET_DATA_ROOT)

# ---------------------------------------------------------------
# 4. Настраиваем Python на правильный репозиторий
# ---------------------------------------------------------------
os.chdir(REPO)

for module_name in list(sys.modules):
    if module_name == "core" or module_name.startswith("core."):
        del sys.modules[module_name]

sys.path = [path for path in sys.path if "error-geometry-routing" not in path]
sys.path.insert(0, str(REPO))

# Это должно сработать: именно здесь раньше была ошибка.
from core.data import get_dataset
from core.models import load_model
from core.paths import DatasetResults

print("✓ Импорты core работают")

# ---------------------------------------------------------------
# 5. Пути сохранения и восстановление прогресса
# ---------------------------------------------------------------
MOUNT = Path("/content/drive")
DRIVE_DIR = MOUNT / "MyDrive" / "error-geometry-routing"
OUT = DRIVE_DIR / "spatial_b0_router_20261028"
SAVED_RESULTS = OUT / "baseline_results"

OUT.mkdir(parents=True, exist_ok=True)

if SAVED_RESULTS.exists() and (SAVED_RESULTS / "cityscapes").exists():
    shutil.copytree(SAVED_RESULTS, REPO / "results", dirs_exist_ok=True)
    print("✓ Восстановлены ранее сохранённые результаты B0")

# ---------------------------------------------------------------
# 6. B0 inference — только для ещё не готовых кадров
# ---------------------------------------------------------------
ds_b0 = get_dataset("cityscapes", full=False)
res_b0 = DatasetResults("cityscapes")

missing = [
    index
    for index, image_id in enumerate(ds_b0.image_ids)
    if not res_b0.mask(image_id).exists()
]

print(f"B0 masks: готово {len(ds_b0) - len(missing)}/{len(ds_b0)}")

if missing:
    model_b0 = load_model("b0")

    drive_masks = SAVED_RESULTS / "cityscapes" / "masks"
    drive_masks.mkdir(parents=True, exist_ok=True)

    for count, index in enumerate(missing, start=1):
        sample = ds_b0[index]
        image_id = sample["image_id"]

        probs = model_b0.probs(sample, ds_b0.size)
        pred_b0 = probs.argmax(dim=0).detach().cpu().numpy().astype(np.uint8)

        mask_file = res_b0.mask(image_id)
        mask_file.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            mask_file,
            pred_b0=pred_b0,
            pred_b2=pred_b0.copy(),
        )

        # Сохранение каждого готового кадра на Drive.
        shutil.copy2(mask_file, drive_masks / mask_file.name)

        if count % 25 == 0 or count == len(missing):
            print(f"B0: {count}/{len(missing)}")

    print("✓ B0 masks сохранены на Drive")
else:
    print("✓ B0 уже полностью рассчитана")

# ---------------------------------------------------------------
# 7. Логиты B0 и кэши для роутера
# ---------------------------------------------------------------
res = DatasetResults("cityscapes")

if not res.logits.exists():
    print("Считаю логиты B0…")
    subprocess.run(
        [sys.executable, "-m", "features.logits", "--dataset", "cityscapes"],
        cwd=REPO,
        check=True,
    )
    print("✓ Логиты B0 готовы")
else:
    print("✓ Логиты B0 уже есть")

from core.evaluation import routing_cache

for pop in ("p0", "p1"):
    if not res.cache(pop).exists():
        routing_cache("cityscapes", pop)
        print(f"✓ Построен cache_{pop}")
    else:
        print(f"✓ cache_{pop} уже существует")

shutil.copytree(REPO / "results", SAVED_RESULTS, dirs_exist_ok=True)

print("\n✓ Базовая подготовка завершена.")
print("Результаты сохранены на Drive:", SAVED_RESULTS)

✓ Использую уже готовый Cityscapes: 500 images, 500 labels
✓ Найдена ветка с кодом: experiments/frozen-router-results-20261025
✓ Cityscapes подключён к исходникам: /content/error-geometry-routing-experiments-frozen-router-results-20261025/data/cityscapes
✓ Импорты core работают
✓ Восстановлены ранее сохранённые результаты B0
B0 masks: готово 500/500
✓ B0 уже полностью рассчитана
✓ Логиты B0 уже есть
✓ cache_p0 уже существует
✓ cache_p1 уже существует

✓ Базовая подготовка завершена.
Результаты сохранены на Drive: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/baseline_results


In [ ]:
# ===== Создание входов spatial router и B5-кэша для P0 =====
# Роутер получает только выход B0: вероятности, энтропию, границу и координаты.
# B5 запускается только на 8 локальных окнах кадра и только для P0.

import numpy as np
import torch
import torch.nn.functional as F

from core.components import NC
from core.data import DATASETS, get_dataset, loader, population
from core.evaluation import _miou, _miou_batch, entropy_score, routing_cache
from core.models import load_model
from core.tiles import grid, tile_effects, tile_index, window
from features.logits import LogitStore

# TILE=256 при рабочем разрешении Cityscapes 1024×512:
# 1024 / 256 = 4 по ширине, 512 / 256 = 2 по высоте, всего 8 тайлов.
TILE = 256
MAP = 16
TOP_K = 3
EPOCHS = 18
SEED = 0

TAG = f"b0spatial_map{MAP}_tile{TILE}_k{TOP_K}"
FEATURE_DIR = OUT / f"{TAG}_features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

size = DATASETS["cityscapes"].size
boxes = grid(size, TILE)
tidx = tile_index(size, TILE)
n_tiles = len(boxes)

expected_n_tiles = (size[0] // TILE) * (size[1] // TILE)
assert n_tiles == expected_n_tiles, (n_tiles, expected_n_tiles, size, TILE)

print(
    f"Сетка B5: {size[0] // TILE} × {size[1] // TILE} "
    f"= {n_tiles} тайлов на кадр"
)

logits = LogitStore("cityscapes")
cache0 = routing_cache("cityscapes", "p0")
cache1 = routing_cache("cityscapes", "p1")

p0_ids = list(population("cityscapes", "p0"))
p1_ids = list(population("cityscapes", "p1"))

print(f"P0: {len(p0_ids)} кадров; P1: {len(p1_ids)} кадров")

# ------------------------------------------------------------------
# B5 на окнах: сохраняется один файл на кадр.
# После остановки GPU повторный запуск продолжит с отсутствующих файлов.
# ------------------------------------------------------------------
B5_DIR = OUT / f"b5_tile_masks_t{TILE}_m128"
B5_DIR.mkdir(parents=True, exist_ok=True)

ds_full = get_dataset("cityscapes", full=True)

@torch.inference_mode()
def ensure_b5(ids):
    missing = [image_id for image_id in ids if not (B5_DIR / f"{image_id}.npy").exists()]

    if not missing:
        print(f"✓ B5 cache уже готов: {len(ids)} кадров")
        return

    print(f"Запускаю B5 на {len(missing)} кадрах P0, по {n_tiles} окон на кадр…")

    model_b5 = load_model("b5")
    wanted = set(missing)

    # Cityscapes: исходный 2048×1024, рабочий 1024×512.
    scale = 2
    margin = 128
    side = TILE + 2 * margin
    side_src = side * scale

    done = 0

    for sample in loader(ds_full):
        image_id = sample["image_id"]

        if image_id not in wanted:
            continue

        src = sample["image"].cuda()
        answer = np.zeros((size[1], size[0]), dtype=np.uint8)

        for x0, y0, x1, y1 in boxes:
            wx, wy = window((x0, y0, x1, y1), TILE, margin, size)

            sx = min(round(wx * scale), src.shape[2] - side_src)
            sy = min(round(wy * scale), src.shape[1] - side_src)

            crop = src[:, sy:sy + side_src, sx:sx + side_src][None]

            pred = model_b5(crop, (side, side))[0].argmax(0).cpu().numpy()

            answer[y0:y1, x0:x1] = pred[
                y0 - wy:y1 - wy,
                x0 - wx:x1 - wx,
            ]

        result_path = B5_DIR / f"{image_id}.npy"
        np.save(result_path, answer)

        done += 1

        if done % 10 == 0 or done == len(missing):
            print(f"B5: {done}/{len(missing)}")

    del model_b5
    torch.cuda.empty_cache()

# B5 запускается пока только для P0: именно здесь роутер обучается и проходит проверку.
ensure_b5(p0_ids)

def b5_mask(image_id):
    path = B5_DIR / f"{image_id}.npy"
    assert path.exists(), f"Не найден B5 результат: {path}"
    return np.load(path)

def boundary(pred):
    """Карта границ классов предсказания B0."""
    edge = np.zeros(pred.shape, dtype=np.float32)
    edge[:, :-1] = np.maximum(edge[:, :-1], pred[:, :-1] != pred[:, 1:])
    edge[:-1, :] = np.maximum(edge[:-1, :], pred[:-1, :] != pred[1:, :])
    return edge

@torch.inference_mode()
def make_feature(image_id, cache):
    """
    Вход роутера до запуска B5:
    - 19 карт вероятностей B0;
    - entropy;
    - разница top-1 и top-2;
    - карта границ B0;
    - x/y координаты;
    - глобальная статистика всего кадра.
    """
    row = cache["pos"][image_id]

    b0 = cache["b0"][row]
    gt = cache["gt"][row]
    b5 = b5_mask(image_id)

    p = logits[image_id].softmax(0)

    ent = entropy_score(p)[None]

    top2 = p.topk(2, dim=0).values
    margin = (top2[0] - top2[1])[None]

    edge = torch.from_numpy(boundary(b0))[None].cuda()

    yy, xx = torch.meshgrid(
        torch.linspace(0, 1, size[1], device="cuda"),
        torch.linspace(0, 1, size[0], device="cuda"),
        indexing="ij",
    )

    # 19 + entropy + margin + boundary + x + y = 24 канала.
    maps = torch.cat([p, ent, margin, edge, xx[None], yy[None]])

    # 19 средних вероятностей + средняя entropy = 20 глобальных признаков.
    global_context = torch.cat(
        [p.mean((1, 2)), ent.mean((1, 2))]
    ).cpu().numpy().astype(np.float16)

    x_tiles = []
    entropy_tiles = []

    for x0, y0, x1, y1 in boxes:
        x_tiles.append(
            F.interpolate(
                maps[:, y0:y1, x0:x1][None],
                size=(MAP, MAP),
                mode="bilinear",
                align_corners=False,
            )[0].cpu().numpy().astype(np.float16)
        )

        entropy_tiles.append(
            float(ent[0, y0:y1, x0:x1].mean().cpu())
        )

    # delta[tile] — насколько B5 улучшает/ухудшает B0 на данном тайле.
    _, _, delta = tile_effects(gt, b0, b5, tidx, n_tiles)

    return {
        "x": np.stack(x_tiles),
        "g": global_context,
        "entropy": np.asarray(entropy_tiles, dtype=np.float32),
        "gain": (delta / (TILE * TILE)).astype(np.float32),
    }

def build_features(ids, cache, population_name):
    for number, image_id in enumerate(ids, start=1):
        path = FEATURE_DIR / f"{population_name}_{image_id}.npz"

        if not path.exists():
            np.savez_compressed(
                path,
                **make_feature(image_id, cache),
            )

        if number % 25 == 0 or number == len(ids):
            print(
                f"Входы router для {population_name}: "
                f"{number}/{len(ids)}"
            )

# Создаём признаки только для P0.
# P1 не используется, пока P0 не подтвердит улучшение над entropy.
build_features(p0_ids, cache0, "p0")

print("\n✓ B5-кэш P0 и входы spatial router готовы.")
print("Папка признаков:", FEATURE_DIR)

Сетка B5: 4 × 2 = 8 тайлов на кадр
P0: 300 кадров; P1: 200 кадров
Запускаю B5 на 300 кадрах P0, по 8 окон на кадр…


config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  339MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  339MB            

model.safetensors: downloading bytes:           |  0.00B            

B5: 10/300
B5: 20/300
B5: 30/300
B5: 40/300
B5: 50/300
B5: 60/300
B5: 70/300
B5: 80/300
B5: 90/300
B5: 100/300
B5: 110/300
B5: 120/300
B5: 130/300
B5: 140/300
B5: 150/300
B5: 160/300
B5: 170/300
B5: 180/300
B5: 190/300
B5: 200/300
B5: 210/300
B5: 220/300
B5: 230/300
B5: 240/300
B5: 250/300
B5: 260/300
B5: 270/300
B5: 280/300
B5: 290/300
B5: 300/300
Входы router для p0: 25/300
Входы router для p0: 50/300
Входы router для p0: 75/300
Входы router для p0: 100/300
Входы router для p0: 125/300
Входы router для p0: 150/300
Входы router для p0: 175/300
Входы router для p0: 200/300
Входы router для p0: 225/300
Входы router для p0: 250/300
Входы router для p0: 275/300
Входы router для p0: 300/300

✓ B5-кэш P0 и входы spatial router готовы.
Папка признаков: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/b0spatial_map16_tile256_k3_features


In [ ]:
# ===== Архитектура роутера и загрузка подготовленных входов =====

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpatialRouter(nn.Module):
    """
    Маленькая сеть-роутер.

    x — карта признаков B0 для одного тайла:
        24 × 16 × 16.
    g — 20 глобальных признаков всего кадра.
    Выход — ожидаемый выигрыш B5 над B0 для тайла.
    """

    def __init__(self, channels=24, global_dim=20):
        super().__init__()

        self.local = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

        self.score = nn.Sequential(
            nn.Linear(32 + global_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x, g):
        local_features = self.local(x)
        features = torch.cat([local_features, g], dim=1)
        return self.score(features).squeeze(1)


def read_records(ids, population_name):
    """Загружает подготовленные в предыдущей ячейке признаки тайлов."""
    records = {}

    for image_id in ids:
        path = FEATURE_DIR / f"{population_name}_{image_id}.npz"

        assert path.exists(), (
            f"Не найден файл признаков: {path}. "
            "Сначала полностью выполните предыдущую ячейку."
        )

        z = np.load(path)
        records[image_id] = {
            key: z[key]
            for key in z.files
        }

    return records


records0 = read_records(p0_ids, "p0")

example = records0[p0_ids[0]]

print("✓ Роутер создан")
print("Признаки одного кадра:")
print("  тайлы x:", example["x"].shape)
print("  глобальный контекст g:", example["g"].shape)
print("  целевой gain:", example["gain"].shape)



✓ Роутер создан
Признаки одного кадра:
  тайлы x: (8, 24, 16, 16)
  глобальный контекст g: (20,)
  целевой gain: (8,)


In [ ]:
# Сохраняем уже готовый результат OOF — обучение повторно не запускается.

import pandas as pd
from pathlib import Path

OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)

p0_report = pd.DataFrame(rows)

report_path = OUT / f"{TAG}_p0_oof.csv"
p0_report.to_csv(report_path, index=False)

display(p0_report)
print("✓ OOF-результат сохранён:", report_path)
print("Разрешённые для проверки на P1 бюджеты:", approved)

,B5_patches_per_frame,method,p0_oof_mIoU,delta_to_entropy,ci_lo,ci_hi,approved_for_P1
0,1,spatial B0 router,74.961330,1.011814,0.276832,2.016417,True
1,2,spatial B0 router,77.409825,0.772969,0.025964,1.763524,True
2,3,spatial B0 router,78.889620,0.690228,0.017078,1.748767,True


✓ OOF-результат сохранён: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/b0spatial_map16_tile256_k3_p0_oof.csv
Разрешённые для проверки на P1 бюджеты: {1: True, 2: True, 3: True}


In [ ]:
# OOF на P0: сравниваем router с entropy при одинаковом числе B5-плиток.
def fit_router(ids, records):
    x = np.concatenate([records[i]['x'] for i in ids]).astype(np.float32)
    g = np.concatenate([np.repeat(records[i]['g'][None], n_tiles, axis=0) for i in ids]).astype(np.float32)
    y = np.concatenate([records[i]['gain'] for i in ids]).astype(np.float32)
    model = SpatialRouter().cuda(); opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-4)
    order = np.arange(len(y)); gen = np.random.default_rng(SEED)
    for _ in range(EPOCHS):
        gen.shuffle(order)
        for start in range(0, len(y), 128):
            ix = order[start:start+128]
            pred = model(torch.from_numpy(x[ix]).cuda(), torch.from_numpy(g[ix]).cuda())
            loss = F.smooth_l1_loss(pred, torch.from_numpy(y[ix]).cuda(), beta=.02)
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()

@torch.inference_mode()
def scores(model, record):
    x = torch.from_numpy(record['x'].astype(np.float32)).cuda()
    g = torch.from_numpy(np.repeat(record['g'][None], len(x), axis=0).astype(np.float32)).cuda()
    return model(x, g).cpu().numpy()

def selected(score, k):
    return np.argsort(-score, kind='stable')[:k]

from sklearn.model_selection import GroupKFold
fold_scores = {}
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(p0_ids, groups=p0_ids), 1):
    model = fit_router(np.asarray(p0_ids)[tr], records0)
    for image_id in np.asarray(p0_ids)[va]: fold_scores[image_id] = scores(model, records0[image_id])
    del model; torch.cuda.empty_cache(); print('OOF fold', fold, '/ 5 готов')

def confs(ids, cache, records, learned):
    out = {k: {'entropy': [], 'spatial-router': []} for k in (1, 2, 3)}
    for image_id in ids:
        row = cache['pos'][image_id]; gt, b0, b5 = cache['gt'][row], cache['b0'][row], b5_mask(image_id)
        d, c0, _ = tile_effects(gt, b0, b5, tidx, n_tiles)
        for k in (1, 2, 3):
            for name, score in [('entropy', records[image_id]['entropy']), ('spatial-router', learned[image_id])]:
                out[k][name].append(c0 + d[selected(score, k)].sum(0))
    return out

oof_conf = confs(p0_ids, cache0, records0, fold_scores)
rng = np.random.default_rng(0); rows, approved = [], {}
for k in (1, 2, 3):
    base, new = np.stack(oof_conf[k]['entropy']), np.stack(oof_conf[k]['spatial-router'])
    w = rng.multinomial(len(base), np.full(len(base), 1/len(base)), size=2000)
    diff = _miou_batch(np.einsum('bn,nij->bij', w, new)) - _miou_batch(np.einsum('bn,nij->bij', w, base))
    delta = _miou(new.sum(0)) - _miou(base.sum(0)); lo, hi = np.percentile(diff, [2.5, 97.5])
    approved[k] = bool(delta > 0 and lo > 0)
    rows.append({'B5_patches_per_frame':k, 'method':'spatial B0 router', 'p0_oof_mIoU':_miou(new.sum(0)),
                 'delta_to_entropy':delta, 'ci_lo':lo, 'ci_hi':hi, 'approved_for_P1':approved[k]})
p0_report = pd.DataFrame(rows); p0_report.to_csv(OUT/f'{TAG}_p0_oof.csv', index=False); display(p0_report)



OOF fold 1 / 5 готов
OOF fold 2 / 5 готов
OOF fold 3 / 5 готов
OOF fold 4 / 5 готов
OOF fold 5 / 5 готов


,B5_patches_per_frame,method,p0_oof_mIoU,delta_to_entropy,ci_lo,ci_hi,approved_for_P1
0,1,spatial B0 router,74.490228,0.540712,0.061485,1.170419,True
1,2,spatial B0 router,77.300281,0.663425,-0.048209,1.622895,False
2,3,spatial B0 router,78.708477,0.509085,0.135098,1.037729,True


In [ ]:
from pathlib import Path
import json
import torch

SHARE_DIR = OUT / "share_spatial_router"
SHARE_DIR.mkdir(parents=True, exist_ok=True)

# final — роутер, дообученный на всех данных Cityscapes P0.
assert "final" in globals(), "Сначала выполните ячейку полного обучения final на P0."

checkpoint_path = SHARE_DIR / "spatial_router_cityscapes_p0.pt"

torch.save(
    {
        "state_dict": {
            key: value.detach().cpu()
            for key, value in final.state_dict().items()
        },
        "architecture": "SpatialRouter",
        "channels": 24,
        "global_dim": 20,
        "tile": 256,
        "margin": 128,
        "trained_on": "Cityscapes P0",
    },
    checkpoint_path,
)

meta = {
    "architecture": "SpatialRouter",
    "channels": 24,
    "global_dim": 20,
    "tile": 256,
    "margin": 128,
    "trained_on": "Cityscapes P0",
    "selection": "top-k predicted B5 gain",
    "note": "Load this checkpoint directly; Cityscapes P0 features are not needed.",
}
(SHARE_DIR / "router_meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Готовая папка для передачи:", SHARE_DIR)
print("Главный файл:", checkpoint_path)

NameError: name 'OUT' is not defined

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json
import torch

# Папка с результатами вашего эксперимента на Google Drive.
OUT = Path(
    "/content/drive/MyDrive/error-geometry-routing/"
    "spatial_b0_router_20261028"
)
OUT.mkdir(parents=True, exist_ok=True)

SHARE_DIR = OUT / "share_spatial_router"
SHARE_DIR.mkdir(parents=True, exist_ok=True)

# final — модель после финального обучения на ВСЕХ данных Cityscapes P0.
# Нельзя подставлять сюда последнюю fold-модель: она обучена только на 4/5 P0.
assert "final" in globals(), (
    "В памяти нет переменной final.\n"
    "Запустите сначала ячейку полного обучения финального роутера на P0, "
    "а затем, не перезапуская runtime, выполните эту ячейку."
)

checkpoint_path = SHARE_DIR / "spatial_router_cityscapes_p0.pt"

torch.save(
    {
        "state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in final.state_dict().items()
        },
        "architecture": "SpatialRouter",
        "channels": 24,
        "global_dim": 20,
        "tile": 256,
        "margin": 128,
        "trained_on": "Cityscapes P0",
        "selection": "top-k predicted B5 gain",
    },
    checkpoint_path,
)

meta = {
    "architecture": "SpatialRouter",
    "channels": 24,
    "global_dim": 20,
    "tile": 256,
    "margin": 128,
    "trained_on": "Cityscapes P0",
    "selection": "top-k predicted B5 gain",
    "load_inference_only": True,
    "note": "P0 features and OOF fold models are not required for ACDC inference.",
}
meta_path = SHARE_DIR / "router_meta.json"
meta_path.write_text(
    json.dumps(meta, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("✓ Роутер сохранён:", checkpoint_path)
print("✓ Метаданные:", meta_path)
print("✓ Передайте другу всю папку:", SHARE_DIR)

Mounted at /content/drive


AssertionError: В памяти нет переменной final.
Запустите сначала ячейку полного обучения финального роутера на P0, а затем, не перезапуская runtime, выполните эту ячейку.

In [ ]:
# Финальное обучение SpatialRouter на уже сохранённых P0-признаках.
# B0 и B5 заново НЕ запускаются.

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DRIVE_ROOT = Path("/content/drive/MyDrive/error-geometry-routing")
assert DRIVE_ROOT.exists(), f"Не найдена папка: {DRIVE_ROOT}"

# Ищем папку с P0-признаками, сохранёнными предыдущим запуском.
feature_dirs = []
for folder in DRIVE_ROOT.rglob("b0spatial_map16_tile256_k3_features"):
    files = sorted(folder.glob("p0_*.npz"))
    if files:
        feature_dirs.append((len(files), folder, files))

assert feature_dirs, (
    "Не найдены p0_*.npz. Нужна папка "
    "b0spatial_map16_tile256_k3_features с сохранёнными P0-признаками."
)

_, FEATURE_DIR, p0_files = max(feature_dirs, key=lambda item: item[0])
OUT = FEATURE_DIR.parent
SHARE_DIR = OUT / "share_spatial_router"
SHARE_DIR.mkdir(parents=True, exist_ok=True)

print("P0-признаки:", FEATURE_DIR)
print("Кадров P0:", len(p0_files))

# Собираем обучающие тайлы.
xs, gs, ys = [], [], []

for path in p0_files:
    data = np.load(path)

    x = data["x"].astype(np.float32)              # [тайлы, 24, 16, 16]
    gain = data["gain"].astype(np.float32).ravel() # целевой выигрыш B5 над B0
    g = data["g"].astype(np.float32)

    assert x.shape[0] == gain.shape[0], (
        f"Некорректный файл {path.name}: "
        f"{x.shape[0]} тайлов, {gain.shape[0]} gain."
    )

    if g.ndim == 1:
        g = np.repeat(g[None, :], x.shape[0], axis=0)
    elif g.shape[0] == 1:
        g = np.repeat(g, x.shape[0], axis=0)

    assert g.shape[0] == x.shape[0], f"Некорректный global context в {path.name}"

    xs.append(x)
    gs.append(g)
    ys.append(gain)

X = np.concatenate(xs, axis=0)
G = np.concatenate(gs, axis=0)
Y = np.concatenate(ys, axis=0)

print(f"Обучающих тайлов: {len(Y)}")
print(f"X: {X.shape}; G: {G.shape}; gain: [{Y.min():.5f}, {Y.max():.5f}]")

# Та же архитектура, что использовалась в OOF-проверке.
class SpatialRouter(nn.Module):
    def __init__(self, channels=24, global_dim=20):
        super().__init__()
        self.local = nn.Sequential(
            nn.Conv2d(channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.score = nn.Sequential(
            nn.Linear(32 + global_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x, g):
        return self.score(torch.cat([self.local(x), g], dim=1)).squeeze(1)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

dataset = TensorDataset(
    torch.from_numpy(X),
    torch.from_numpy(G),
    torch.from_numpy(Y),
)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

final = SpatialRouter(
    channels=X.shape[1],
    global_dim=G.shape[1],
).to(device)

optimizer = torch.optim.AdamW(
    final.parameters(),
    lr=2e-3,
    weight_decay=2e-4,
)
loss_fn = nn.SmoothL1Loss(beta=0.02)

final.train()
for epoch in range(1, 19):
    total_loss = 0.0

    for xb, gb, yb in loader:
        xb, gb, yb = xb.to(device), gb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        prediction = final(xb, gb)
        loss = loss_fn(prediction, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)

    print(f"Epoch {epoch:02d}/18 | loss = {total_loss / len(dataset):.6f}")

final.eval()

# Сохраняем финальную модель и метаданные для другого человека.
checkpoint_path = SHARE_DIR / "spatial_router_cityscapes_p0.pt"
torch.save(
    {
        "state_dict": {
            name: value.detach().cpu()
            for name, value in final.state_dict().items()
        },
        "architecture": "SpatialRouter",
        "channels": int(X.shape[1]),
        "global_dim": int(G.shape[1]),
        "tile": 256,
        "margin": 128,
        "trained_on": "Cityscapes P0",
        "selection": "top-k predicted B5 gain",
    },
    checkpoint_path,
)

meta_path = SHARE_DIR / "router_meta.json"
meta_path.write_text(
    json.dumps(
        {
            "architecture": "SpatialRouter",
            "channels": int(X.shape[1]),
            "global_dim": int(G.shape[1]),
            "tile": 256,
            "margin": 128,
            "trained_on": "Cityscapes P0",
            "selection": "top-k predicted B5 gain",
            "load_inference_only": True,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n✓ Финальный роутер сохранён")
print("Checkpoint:", checkpoint_path)
print("Metadata: ", meta_path)
print("Передайте другу папку:", SHARE_DIR)

P0-признаки: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/b0spatial_map16_tile256_k3_features
Кадров P0: 300
Обучающих тайлов: 2400
X: (2400, 24, 16, 16); G: (2400, 20); gain: [-0.34282, 0.35643]
Epoch 01/18 | loss = 0.039218
Epoch 02/18 | loss = 0.008940
Epoch 03/18 | loss = 0.006437
Epoch 04/18 | loss = 0.005967
Epoch 05/18 | loss = 0.005823
Epoch 06/18 | loss = 0.005785
Epoch 07/18 | loss = 0.005636
Epoch 08/18 | loss = 0.005550
Epoch 09/18 | loss = 0.005562
Epoch 10/18 | loss = 0.005591
Epoch 11/18 | loss = 0.005614
Epoch 12/18 | loss = 0.005724
Epoch 13/18 | loss = 0.005602
Epoch 14/18 | loss = 0.005551
Epoch 15/18 | loss = 0.005537
Epoch 16/18 | loss = 0.005527
Epoch 17/18 | loss = 0.005497
Epoch 18/18 | loss = 0.005514

✓ Финальный роутер сохранён
Checkpoint: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/share_spatial_router/spatial_router_cityscapes_p0.pt
Metadata:  /content/drive/MyDrive/error-geometry-routing/spatial_b0_r

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

DRIVE_ROOT = Path("/content/drive/MyDrive/error-geometry-routing")

share_dirs = list(DRIVE_ROOT.rglob("share_spatial_router"))
assert share_dirs, (
    "Не найдена папка share_spatial_router. "
    "Сначала выполните ячейку сохранения финального роутера."
)

SHARE_DIR = max(
    share_dirs,
    key=lambda folder: (folder / "spatial_router_cityscapes_p0.pt").exists()
)

checkpoint = SHARE_DIR / "spatial_router_cityscapes_p0.pt"
metadata = SHARE_DIR / "router_meta.json"

assert checkpoint.exists(), f"Не найден checkpoint: {checkpoint}"
assert metadata.exists(), f"Не найден файл метаданных: {metadata}"

ZIP_BASE = SHARE_DIR.parent / "spatial_router_cityscapes_p0_share"
ZIP_PATH = Path(
    shutil.make_archive(
        base_name=str(ZIP_BASE),
        format="zip",
        root_dir=str(SHARE_DIR),
    )
)

print("✓ ZIP готов:", ZIP_PATH)
print(f"✓ Размер: {ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MB")
print("Начинаю скачивание...")

files.download(str(ZIP_PATH))

✓ ZIP готов: /content/drive/MyDrive/error-geometry-routing/spatial_b0_router_20261028/spatial_router_cityscapes_p0_share.zip
✓ Размер: 0.07 MB
Начинаю скачивание...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>